## 1 · Imports

In [ ]:
%matplotlib inline

import logging
import os
import time
import warnings
from dataclasses import asdict, dataclass, replace
from typing import Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_auc_score
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    RandomizedSearchCV,
    StratifiedGroupKFold,
)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor

plt.rcParams["font.family"] = ["Liberation Sans", "Arimo", "DejaVu Sans"]
plt.rcParams["svg.fonttype"] = "none"

logging.basicConfig(format="%(levelname)s | %(message)s", level=logging.INFO)
log = logging.getLogger("pipeline")


class PipelineError(Exception):
    """Raised when data or config make a pipeline run impossible."""

## 2 · Configuration

Everything tunable lives here. Key knobs:

- `task`: `"binary"` (AUC) or `"continuous"` (C-index)
- `cap_age`: keep a pair if **either** member is below the cap (`None` = no filter)
- `min_pairs`: skip the analysis if fewer complete pairs survive
- `downsample`: pair-level balancing, **seeded**, applied to each **train split only**
- `tune`: light hyperparameter search on/off (off = vetted defaults)
- `ci_methods`: any subset of `("runs", "predictions", "refit")`
- `n_runs`, `n_boot`, `n_refit`, `n_perm`: budget knobs — lower = faster

In [ ]:
@dataclass
class PipelineConfig:
    """Every knob of the pipeline in one place."""

    task: Literal["binary", "continuous"] = "binary"
    seed: int = 42
    # column names / feature detection
    pair_col: str = "new_par"
    age_col: str = "age_at_diagnosis"
    #feature_regex: str = r"^X\d+$"
    feature_regex: str = r"^Calc"
    # preprocessing
    cap_age: float | None = None      # keep pair if EITHER member is below cap (binary); row-wise for continuous
    min_pairs: int = 5               # minimum complete pairs required, else skip
    downsample: bool = True           # binary only; applied inside each TRAIN split only
    # repeated hold-out
    n_runs: int = 30
    test_size: float = 0.3
    # model / tuning
    tune: bool = True
    n_search_iter: int = 10
    inner_cv_splits: int = 5
    # confidence intervals
    ci_methods: tuple[str, ...] = ("runs", "predictions", "refit")
    ci_level: float = 0.95
    n_boot: int = 1000                # draws for 'runs' and 'predictions'
    n_refit: int = 500                # draws for 'refit' (1 split per draw)
    refit_tune: bool = False          # tune inside refit draws (slow); default reuses consensus params
    # permutation test
    n_perm: int = 10000
    # feature importance
    importance: bool = False
    top_n_features: int = 20
    n_boot_importance: int = 1000
    feature_name_map: dict | None = None
    # execution
    n_jobs: int = -1

    def check(self) -> None:
        if self.task not in ("binary", "continuous"):
            raise PipelineError(f"task must be 'binary' or 'continuous', got {self.task!r}")
        if not 0 < self.test_size < 1:
            raise PipelineError(f"test_size must be in (0, 1), got {self.test_size}")
        if self.n_runs < 2:
            raise PipelineError(f"n_runs must be >= 2, got {self.n_runs}")
        unknown = set(self.ci_methods) - {"runs", "predictions", "refit"}
        if unknown:
            raise PipelineError(f"unknown ci_methods {unknown}; valid: runs, predictions, refit")


def spawn_seeds(seed: int, n: int) -> list[int]:
    """n independent child seeds from one master seed — the single source of randomness."""
    return [int(s.generate_state(1)[0]) for s in np.random.SeedSequence(seed).spawn(n)]

## 3 · Validation gate

Runs before any modelling: required columns, feature block, target type, class counts,
pair integrity, constant/NaN features. **FAIL stops the pipeline with a clear message** —
this is the first place to look when something is off.

In [ ]:
def validate_data(df: pd.DataFrame, cfg: PipelineConfig, target: str) -> pd.DataFrame:
    """Gate before any modelling. Prints a PASS/WARN/FAIL table; raises on FAIL."""
    checks = []

    def add(name, ok, detail="", level="FAIL"):
        checks.append({"check": name, "status": "PASS" if ok else level, "detail": detail})

    for col in (cfg.pair_col, cfg.age_col, target):
        add(f"column '{col}' present", col in df.columns)
    feature_cols = _feature_columns(df, cfg)
    add("feature columns found", len(feature_cols) > 0, f"{len(feature_cols)} columns match {cfg.feature_regex}")
    if target in df.columns:
        add("no NaN in target", df[target].notna().all(), f"{df[target].isna().sum()} NaN")
        if cfg.task == "binary":
            vc = df[target].value_counts()
            add("target is binary 0/1", set(df[target].unique()) <= {0, 1}, f"classes: {dict(vc)}")
            add("both classes present", vc.size == 2, f"counts: {dict(vc)}")
    if cfg.pair_col in df.columns:
        sizes = df.groupby(cfg.pair_col).size()
        add("pair sizes <= 2", (sizes <= 2).all(), f"max pair size {sizes.max()}", level="WARN")
    if feature_cols:
        n_const = int((df[feature_cols].std() == 0).sum())
        add("no constant features", n_const == 0, f"{n_const} constant features", level="WARN")
        add("no NaN in features", not df[feature_cols].isna().any().any(), level="WARN")

    table = pd.DataFrame(checks)
    print(table.to_string(index=False))
    if (table["status"] == "FAIL").any():
        raise PipelineError("validation failed — fix the FAIL rows above before running")
    return table


def _feature_columns(df: pd.DataFrame, cfg: PipelineConfig) -> list[str]:
    cols = df.columns[df.columns.str.match(cfg.feature_regex)].tolist()
    try:
        return sorted(cols, key=lambda c: int(c[1:]))  # X1, X2, ..., X1350
    except ValueError:
        return sorted(cols)  # non-numeric suffixes: plain sort

## 4 · Preprocessing & downsampling

- Age cap (either-member rule) and `min_pairs` enforcement.
- `downsample_train`: exact pair-level class balance — keeps all pairs containing the minority
  class, tops up with seeded-random majority-only pairs. **Train split only**: the test set
  always keeps the natural prevalence, so metrics stay honest.

In [ ]:
@dataclass
class DataSet:
    """Array view of the modelling table — compact and cheap to ship to workers."""

    X: np.ndarray          # (n_rows, n_features) float32
    y: np.ndarray          # (n_rows,)
    pairs: np.ndarray      # (n_rows,) int-coded pair id
    feature_names: list[str]

    @property
    def n_pairs(self) -> int:
        return int(np.unique(self.pairs).size)


def preprocess(
    df: pd.DataFrame,
    cfg: PipelineConfig,
    target: str
) -> tuple[DataSet | None, dict]:
    """Preprocess data for modelling. Unmatched pairs are allowed."""

    # ============================================================
    # 1. INITIALISE REPORT
    #    Record the number of rows and pairs before preprocessing.
    # ============================================================
    report = {
        "issues": [],
        "rows_in": len(df),
        "pairs_in": df[cfg.pair_col].nunique(),
    }

    print("\nPreprocessing:")
    print(f"  Rows entering:            {len(df)}")
    print(f"  Pairs entering:           {df[cfg.pair_col].nunique()}")
    print(f"  Target distribution:")
    print(df[target].value_counts(dropna=False).to_string())


    # ============================================================
    # 2. AGE CAP
    #    Keep only pairs that have at least one observation
    #    below the specified age cap.
    # ============================================================
    if cfg.cap_age is not None:
        ok_pairs = df.loc[
            df[cfg.age_col] < cfg.cap_age,
            cfg.pair_col
        ].unique()

        df = df[df[cfg.pair_col].isin(ok_pairs)]
        df = df.reset_index(drop=True)

    report["rows_after_age_cap"] = len(df)
    report["pairs_after_age_cap"] = df[cfg.pair_col].nunique()

    print("\nAfter age cap:")
    print(f"  Rows:                     {len(df)}")
    print(f"  Pairs:                    {df[cfg.pair_col].nunique()}")
    print(f"  Target distribution:")
    print(df[target].value_counts(dropna=False).to_string())


    # ============================================================
    # 3. IDENTIFY FEATURES
    #    Find the Calc columns that will be used as predictors.
    #    NaNs in these columns are allowed because XGBoost can
    #    handle missing feature values.
    # ============================================================
    feats = _feature_columns(df, cfg)

    print("\nFeatures:")
    print(f"  Number of features:       {len(feats)}")
    print(f"  Feature columns:           {feats}")


    # ============================================================
    # 4. REMOVE ROWS WITH NaN IN CALC FEATURES
    #    Remove only rows where at least one Calc feature is NaN.
    #    Do not remove rows based on target, pair ID, or age.
    # ============================================================
    n_before_dropna = len(df)

    df = df.dropna(subset=feats).reset_index(drop=True)

    n_removed = n_before_dropna - len(df)

    report["rows_after_dropna"] = len(df)
    report["rows_removed_nan"] = n_removed
    report["pairs_after_dropna"] = df[cfg.pair_col].nunique()

    print("\nAfter removing rows with NaN in Calc features:")
    print(f"  Rows before:              {n_before_dropna}")
    print(f"  Rows remaining:           {len(df)}")
    print(f"  Rows removed:             {n_removed}")
    print(f"  Pairs remaining:          {df[cfg.pair_col].nunique()}")
    print("  Target distribution:")
    print(df[target].value_counts(dropna=False).to_string())


    # ============================================================
    # 5. FEATURE MISSINGNESS
    #    Show how many cases and controls have a value for each
    #    Calc feature. These NaNs are NOT removed.
    # ============================================================
    n_controls = (df[target] == 0).sum()
    n_cases = (df[target] == 1).sum()

    feature_counts = pd.DataFrame({
        "controls_n": df.loc[df[target] == 0, feats].notna().sum(),
        "cases_n": df.loc[df[target] == 1, feats].notna().sum(),
    })

    feature_counts["controls_%"] = (
        feature_counts["controls_n"] / n_controls * 100
    ).round(1)

    feature_counts["cases_%"] = (
        feature_counts["cases_n"] / n_cases * 100
    ).round(1)

    feature_counts["controls_missing"] = (
        n_controls - feature_counts["controls_n"]
    )

    feature_counts["cases_missing"] = (
        n_cases - feature_counts["cases_n"]
    )

    print("\nCalc feature availability by target:")
    print(feature_counts.to_string())

    print(
        f"\nTotal NaNs remaining in Calc features: "
        f"{df[feats].isna().sum().sum()}"
    )




    # ============================================================
    # 6. FINAL CHECK
    #    Stop if no usable rows remain.
    # ============================================================
    if len(df) == 0:
        report["issues"].append(
            "no rows remaining after removing invalid rows"
        )
        return None, report


    # ============================================================
    # 7. CREATE DATASET
    #    Convert the cleaned dataframe into the format used by
    #    the modelling pipeline.
    # ============================================================
    data = DataSet(
        X=df[feats].to_numpy(np.float32),
        y=df[target].to_numpy(),
        pairs=pd.factorize(df[cfg.pair_col])[0].astype(np.int64),
        feature_names=feats,
    )

    report["n_features"] = len(feats)

    return data, report


def downsample_train(train_idx: np.ndarray, y: np.ndarray, pairs: np.ndarray, rng: np.random.Generator) -> tuple[np.ndarray, dict]:
    """Exact pair-level class balance on the TRAIN set only.

    Keeps every pair containing a minority-class member, then tops up with
    randomly chosen majority-only pairs so class counts match exactly
    (when majority-only pairs are single-row). Seeded -> reproducible.
    """
    report: dict = {"issues": []}
    y_t, p_t = y[train_idx], pairs[train_idx]
    n_pos, n_neg = int(y_t.sum()), int((y_t == 0).sum())
    report.update(train_before=len(train_idx), cases=n_pos, controls=n_neg)
    if n_pos == n_neg:
        report["downsampled"] = False
        return train_idx, report

    minority = 1 if n_pos < n_neg else 0
    pair_has_min = pd.DataFrame({"p": p_t, "y": y_t}).groupby("p")["y"].apply(
        lambda s: bool((s == minority).any())
    )
    min_pair_ids = pair_has_min.index[pair_has_min].to_numpy()
    maj_only_ids = pair_has_min.index[~pair_has_min].to_numpy()

    n_min_rows = int((y_t == minority).sum())
    n_maj_kept = int(((y_t != minority) & np.isin(p_t, min_pair_ids)).sum())
    n_needed = n_min_rows - n_maj_kept

    if n_needed > len(maj_only_ids):
        report["issues"].append(
            f"cannot fully balance: need {n_needed} majority-only pairs, have {len(maj_only_ids)}"
        )
        chosen = maj_only_ids
    elif n_needed > 0:
        chosen = rng.permutation(maj_only_ids)[:n_needed]
    else:
        chosen = np.array([], dtype=maj_only_ids.dtype)

    keep = np.isin(p_t, np.concatenate([min_pair_ids, chosen]))
    new_idx = train_idx[keep]
    y_new = y[new_idx]
    report.update(
        downsampled=True,
        train_after=len(new_idx),
        cases_after=int(y_new.sum()),
        controls_after=int((y_new == 0).sum()),
    )
    return new_idx, report

## 5 · Metrics

One metric per task: **ROC AUC** (binary) and **concordance index** (continuous).
The C-index is ties-correct, O(n log n), and reduces exactly to AUC for binary outcomes —
the self-test proves both properties against a brute-force reference and sklearn.

In [ ]:
def _inversions(a: np.ndarray) -> int:
    """Strict inversions (i < j, a[i] > a[j]) via merge sort — O(n log n)."""
    a = np.asarray(a, dtype=np.float64)
    if len(a) < 2:
        return 0
    mid = len(a) // 2
    left, right = a[:mid].copy(), a[mid:].copy()  # copies: merge writes back into a
    inv = _inversions(left) + _inversions(right)
    i = j = k = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            a[k] = left[i]
            i += 1
        else:
            a[k] = right[j]
            inv += len(left) - i
            j += 1
        k += 1
    a[k:] = np.concatenate([left[i:], right[j:]])
    return inv


def concordance_index(y: np.ndarray, p: np.ndarray) -> float:
    """Ties-correct concordance index for uncensored outcomes, O(n log n).

    C = P(pred higher for the observation with the higher true value), ties in
    pred count 1/2, pairs tied in y are excluded. Equals ROC AUC for binary y.
    """
    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    n = len(y)
    order = np.lexsort((p, y))  # sort by y, then by p within y-ties
    y_s, p_s = y[order], p[order]

    total = n * (n - 1) // 2
    inv = _inversions(p_s.copy())

    _, pc = np.unique(p_s, return_counts=True)
    peq_all = int((pc * (pc - 1) // 2).sum())          # all pairs tied in p
    plt_all = total - inv - peq_all                    # all pairs with p_i < p_j

    starts = np.concatenate(([0], np.flatnonzero(np.diff(y_s) != 0) + 1))
    ends = np.concatenate((starts[1:], [n]))
    ytie = peq_within = 0
    for s, e in zip(starts, ends):
        m = e - s
        ytie += m * (m - 1) // 2
        _, cc = np.unique(p_s[s:e], return_counts=True)
        peq_within += int((cc * (cc - 1) // 2).sum())
    plt_within = ytie - peq_within                     # y-tied pairs with p_i < p_j

    comparable = total - ytie
    if comparable == 0:
        return np.nan
    concordant = plt_all - plt_within
    ties = peq_all - peq_within
    return float((concordant + 0.5 * ties) / comparable)


def concordance_brute(y: np.ndarray, p: np.ndarray) -> float:
    """O(n^2) reference implementation — ground truth for testing concordance_index."""
    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    num = den = 0.0
    for i in range(len(y)):
        for j in range(i + 1, len(y)):
            if y[i] == y[j]:
                continue
            den += 1
            s = np.sign(y[i] - y[j]) * np.sign(p[i] - p[j])
            num += 1.0 if s > 0 else (0.5 if s == 0 else 0.0)
    return num / den if den else np.nan


def score_metric(task: str, y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """The single evaluation metric: ROC AUC (binary) or C-index (continuous)."""
    if task == "binary":
        if np.unique(y_true).size < 2:
            return np.nan
        return float(roc_auc_score(y_true, y_pred))
    return concordance_index(y_true, y_pred)

## 6 · Model

XGBoost behind one interface (`XGBClassifier` / `XGBRegressor`). With `tune=True`, a light
`RandomizedSearchCV` runs under **group-aware inner CV** (a pair never straddles inner folds).
`consensus_params` distils the per-run winners into one fixed parameter set for the refit bootstrap.

In [ ]:
PARAM_DIST = {
    "n_estimators": [100, 300, 500],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0],
    "min_child_weight": [1, 5, 10],
    "gamma": [0, 1, 5],
}
DEFAULT_PARAMS = dict(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0,
)
_INT_PARAMS = {"n_estimators", "max_depth", "min_child_weight", "gamma"}


def _make_estimator(task: str, seed: int, params: dict):
    common = dict(random_state=seed, n_jobs=1, tree_method="hist", **params)
    if task == "binary":
        return XGBClassifier(eval_metric="logloss", **common)
    return XGBRegressor(**common)


def fit_model(
    X: np.ndarray,
    y: np.ndarray,
    pairs: np.ndarray,
    cfg: PipelineConfig,
    seed: int,
    fixed_params: dict | None = None,
    tune: bool | None = None,):
    """Fit XGBoost with light grouped-CV search, or directly with fixed/default params."""
    tune = cfg.tune if tune is None else tune
    params = dict(fixed_params) if fixed_params else dict(DEFAULT_PARAMS)
    if not tune:
        model = _make_estimator(cfg.task, seed, params)
        model.fit(X, y)
        return model, params

    n_groups = np.unique(pairs).size
    if cfg.task == "binary":
        min_class = int(np.bincount(y.astype(int), minlength=2).min())
        n_splits = max(2, min(cfg.inner_cv_splits, min_class, n_groups))
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        scoring = "roc_auc"
    else:
        n_splits = max(2, min(cfg.inner_cv_splits, n_groups))
        cv = GroupKFold(n_splits=n_splits)
        scoring = "neg_root_mean_squared_error"

    search = RandomizedSearchCV(
        _make_estimator(cfg.task, seed, {}),
        PARAM_DIST,
        n_iter=cfg.n_search_iter,
        scoring=scoring,
        cv=cv,
        random_state=seed,
        n_jobs=1,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*least populated class.*")
        search.fit(X, y, groups=pairs)
    return search.best_estimator_, dict(search.best_params_)


def consensus_params(best_params_list: list[dict]) -> dict:
    """Per-parameter median across runs -> fixed params for the refit bootstrap."""
    if not best_params_list:
        return dict(DEFAULT_PARAMS)
    df = pd.DataFrame(best_params_list)
    out = {}
    for col in df.columns:
        med = df[col].median()
        out[col] = int(round(med)) if col in _INT_PARAMS else float(med)
    return out

## 7 · Splits & single-run evaluation

Repeated group-aware hold-out (`GroupShuffleSplit` on `new_par`). Per split: downsample train →
fit scaler on train only → fit → score. Every split returns a dict with the metric **or** the
failure reason — nothing is dropped silently.

In [ ]:
def make_splits(data: DataSet, cfg: PipelineConfig, seeds: list[int]):
    """Repeated group-aware hold-out: a pair never straddles train/test."""
    splits = []
    for s in seeds:
        gss = GroupShuffleSplit(n_splits=1, test_size=cfg.test_size, random_state=s)
        tr, te = next(gss.split(data.X, data.y, groups=data.pairs))
        splits.append((tr, te))
    return splits

def evaluate_split(
    *,
    data: DataSet,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    cfg: PipelineConfig,
    seed: int,
    fixed_params: dict | None = None,
    tune: bool | None = None,
    want_importance: bool = True,) -> dict:
    """One train/test split: downsample (train only) -> scale -> fit -> score.

    Never silently drops: failures come back as metric=NaN plus a reason in issues.
    """
    out = dict(
        metric=np.nan, n_train=0, n_test=len(test_idx), best_params={},
        gain={}, issues=[], y_true=None, y_pred=None,
    )
    try:
        rng = np.random.default_rng(seed)
        if cfg.task == "binary" and cfg.downsample:
            train_idx, rep = downsample_train(train_idx, data.y, data.pairs, rng)
            out["issues"] += rep["issues"]
            out["n_train_balanced"] = len(train_idx)
        out["n_train"] = len(train_idx)

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(data.X[train_idx])
        Xte = scaler.transform(data.X[test_idx])

        model, params = fit_model(
            Xtr, data.y[train_idx], data.pairs[train_idx], cfg, seed, fixed_params, tune
        )
        pred = (
            model.predict_proba(Xte)[:, 1]
            if cfg.task == "binary"
            else model.predict(Xte)
        )
        m = score_metric(cfg.task, data.y[test_idx], pred)
        if not np.isfinite(m):
            out["issues"].append("degenerate test set (single class / no comparable pairs)")
            return out
        out.update(
            metric=m, y_true=data.y[test_idx], y_pred=pred, best_params=params
        )
        if want_importance:
            gain = model.get_booster().get_score(importance_type="gain")
            out["gain"] = {
                data.feature_names[int(k[1:])]: v for k, v in gain.items()
            }
    except Exception as e:  # noqa: BLE001 - reported, never silent
        out["issues"].append(f"{type(e).__name__}: {e}")
    return out

## 8 · Bootstrap confidence intervals

Three complementary CIs, any subset selectable:

| method | what varies | what it captures |
|---|---|---|
| `runs` | resample per-run metrics | split-to-split variability |
| `predictions` | resample pooled out-of-sample (y, pred) | test-sample uncertainty, model fixed |
| `refit` | cluster-bootstrap pairs, **refit**, score **out-of-bag** pairs only | full training-procedure variability, no leakage |

The refit CI does **1 fit per draw** with consensus params — B cheap fits, not B × n_runs searches.

In [ ]:
def _percentile_ci(values: np.ndarray, level: float) -> tuple[float, float]:
    a = (1 - level) / 2 * 100
    return float(np.percentile(values, a)), float(np.percentile(values, 100 - a))


def _bootstrap_mean(values: np.ndarray, seeds: list[int]) -> np.ndarray:
    v = np.asarray(values, dtype=float)
    out = np.empty(len(seeds))
    for i, s in enumerate(seeds):
        rng = np.random.default_rng(s)
        out[i] = v[rng.integers(0, len(v), len(v))].mean()
    return out


def ci_runs(metrics: np.ndarray, cfg: PipelineConfig, seeds: list[int]):
    """CI 1 — bootstrap over per-run metrics: split-to-split variability."""
    vals = _bootstrap_mean(metrics, seeds)
    return vals, _percentile_ci(vals, cfg.ci_level)


def ci_predictions(y_true: np.ndarray, y_pred: np.ndarray, cfg: PipelineConfig, seeds: list[int]):
    """CI 2 — bootstrap pooled out-of-sample (y, pred) pairs, model held fixed."""
    n = len(y_true)
    vals = np.empty(len(seeds))
    for i, s in enumerate(seeds):
        rng = np.random.default_rng(s)
        idx = rng.integers(0, n, n)
        vals[i] = score_metric(cfg.task, y_true[idx], y_pred[idx])
    vals = vals[np.isfinite(vals)]
    return vals, _percentile_ci(vals, cfg.ci_level)


def _refit_chunk(seeds: list[int], data: DataSet, cfg: PipelineConfig, fixed_params: dict | None):
    uniq = np.unique(data.pairs)
    rows_for = [np.flatnonzero(data.pairs == p) for p in uniq]
    vals = []
    for s in seeds:
        rng = np.random.default_rng(s)
        draw = rng.integers(0, len(uniq), len(uniq))  # cluster bootstrap over pairs
        in_bag = np.zeros(len(uniq), dtype=bool)
        in_bag[np.unique(draw)] = True
        # train on the bootstrap sample (copies kept, multiplicity intact),
        # test ONLY on out-of-bag pairs -> no individual seen in training
        tr = np.concatenate([rows_for[k] for k in draw])
        te = np.concatenate([rows_for[k] for k in np.flatnonzero(~in_bag)])
        tr_pairs = np.repeat(np.arange(len(draw)), [len(rows_for[k]) for k in draw])
        te_pairs = np.arange(len(draw), len(draw) + len(np.flatnonzero(~in_bag)))
        te_pairs = np.repeat(te_pairs, [len(rows_for[k]) for k in np.flatnonzero(~in_bag)])
        bdata = DataSet(
            X=np.vstack([data.X[tr], data.X[te]]),
            y=np.concatenate([data.y[tr], data.y[te]]),
            pairs=np.concatenate([tr_pairs, te_pairs]),
            feature_names=data.feature_names,
        )
        r = evaluate_split(
            data=bdata,
            train_idx=np.arange(len(tr)),
            test_idx=np.arange(len(tr), len(tr) + len(te)),
            cfg=cfg, seed=s,
            fixed_params=fixed_params, tune=cfg.refit_tune, want_importance=False,
        )
        vals.append(r["metric"])
    return vals


def ci_refit(data: DataSet, cfg: PipelineConfig, fixed_params: dict | None, seeds: list[int]):
    """CI 3 — full-pipeline bootstrap with out-of-bag evaluation.

    Each draw resamples pairs with replacement, refits on the bootstrap sample,
    and scores ONLY on out-of-bag pairs (never seen in training) — no leakage.
    One fit per draw (B = n_refit) with fixed consensus params by default, so the
    cost is B cheap fits, not B x n_runs searches.
    """
    n_chunks = min(len(seeds), (cfg.n_jobs if cfg.n_jobs > 0 else os.cpu_count()) or 1)
    chunks = [c for c in np.array_split(np.asarray(seeds), n_chunks) if len(c)]
    results = Parallel(n_jobs=cfg.n_jobs, backend="loky")(
        delayed(_refit_chunk)(list(map(int, c)), data, cfg, fixed_params) for c in chunks
    )
    vals = np.array([v for r in results for v in r], dtype=float)
    n_failed = int((~np.isfinite(vals)).sum())
    vals = vals[np.isfinite(vals)]
    info = {"n_draws": len(seeds), "n_failed": n_failed}
    if len(vals) < 10:
        raise PipelineError(f"refit bootstrap: only {len(vals)}/{len(seeds)} draws succeeded")
    return vals, _percentile_ci(vals, cfg.ci_level), info

## 9 · Permutation test

Label shuffle within each run's test set (models fixed), null = distribution of the mean
per-run metric. `p = (1 + #{null ≥ observed}) / (1 + n_perm)` — never zero, and computed
per-run so pooled duplicate observations can't make it anti-conservative.

In [ ]:
def permutation_test(run_results: list[dict], cfg: PipelineConfig, seeds: list[int]):
    """Label-shuffle null for the mean per-run metric (models held fixed).

    Per permutation: shuffle y within each run's test set, recompute, average.
    p = (1 + #{null >= observed}) / (1 + n_perm)  — never zero, never anti-conservative
    from pooled duplicate observations.
    """
    valid = [r for r in run_results if r["y_true"] is not None]
    obs = float(np.mean([r["metric"] for r in valid]))
    nulls = np.empty(len(seeds))
    for i, s in enumerate(seeds):
        rng = np.random.default_rng(s)
        nulls[i] = np.mean(
            [score_metric(cfg.task, rng.permutation(r["y_true"]), r["y_pred"]) for r in valid]
        )
    p = (1 + int((nulls >= obs).sum())) / (1 + len(nulls))
    return obs, nulls, float(p)

## 10 · Feature importance

XGBoost gain per run, aggregated across runs with a bootstrap CI on the mean.
`feature_name_map` (e.g. metabolite names) is applied to the displayed table.

In [ ]:
def aggregate_importance(
    gain_dicts: list[dict], feature_names: list[str], cfg: PipelineConfig, seeds: list[int]):
    """Mean gain per feature across runs + bootstrap CI over runs."""
    mat = pd.DataFrame(gain_dicts, columns=feature_names).fillna(0.0)

    def _one_feature(j):
        col = mat.iloc[:, j].to_numpy()
        boots = _bootstrap_mean(col, seeds)
        lo, hi = _percentile_ci(boots, cfg.ci_level)
        return feature_names[j], float(col.mean()), lo, hi

    rows = Parallel(n_jobs=cfg.n_jobs, backend="loky")(
        delayed(_one_feature)(j) for j in range(len(feature_names))
    )
    imp = pd.DataFrame(rows, columns=["feature", "mean_gain", "ci_lower", "ci_upper"])
    imp = imp.sort_values("mean_gain", ascending=False).reset_index(drop=True)
    if cfg.feature_name_map:
        imp["metabolite"] = imp["feature"].map(cfg.feature_name_map)
    return imp.head(cfg.top_n_features), mat


def plot_importance(imp: pd.DataFrame, title: str, path: str | None = None):
    labels = imp["metabolite"] if "metabolite" in imp.columns else imp["feature"]
    fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(imp))))
    y_pos = np.arange(len(imp))
    means = imp["mean_gain"].to_numpy()
    ax.barh(
        y_pos, means,
        xerr=[means - imp["ci_lower"].to_numpy(), imp["ci_upper"].to_numpy() - means],
        color="#4C72B0", capsize=3,
    )
    ax.set_yticks(y_pos, labels)
    ax.invert_yaxis()
    ax.set_xlabel("mean gain")
    ax.set_title(title)
    fig.tight_layout()
    if path:
        fig.savefig(path)
    return fig


def plot_confusion(y_true: np.ndarray, y_pred: np.ndarray, title: str, path: str | None = None):
    cm = confusion_matrix(y_true, (y_pred >= 0.5).astype(int))
    disp = ConfusionMatrixDisplay(cm, display_labels=["control", "case"])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    disp.plot(cmap=plt.cm.Blues, ax=ax, colorbar=False)
    ax.set_title(title)
    fig.tight_layout()
    if path:
        fig.savefig(path)
    return fig

## 11 · Main pipeline

`run_pipeline(df, cfg, target)` → one structured result dict: point estimates, all CIs,
permutation p, per-run table, best params, importance, **issues list**, per-stage timings, config.

In [ ]:
def run_pipeline(df: pd.DataFrame, cfg: PipelineConfig, target: str) -> dict | None:
    """Full pipeline for one trait at one age cap. Returns a structured result dict."""
    t0 = time.perf_counter()
    cfg.check()
    timings: dict = {}
    issues: list[str] = []

    log.info("1/6 validate")
    validate_data(df, cfg, target)  # raises on FAIL

    log.info("2/6 preprocess")
    data, rep = preprocess(df, cfg, target)
    issues += rep["issues"]
    if data is None:
        log.warning("preprocessing left no usable data — skipped (%s)", rep["issues"])
        return None
    log.info("    %d rows, %d pairs, %d features", len(data.y), data.n_pairs, len(data.feature_names))

    log.info("3/6 %d repeated hold-out runs", cfg.n_runs)
    run_seeds = spawn_seeds(cfg.seed, cfg.n_runs)
    splits = make_splits(data, cfg, run_seeds)
    t = time.perf_counter()
    results = Parallel(n_jobs=cfg.n_jobs, backend="loky")(
        delayed(evaluate_split)(
            data=data, train_idx=tr, test_idx=te, cfg=cfg, seed=s,
            want_importance=cfg.importance,
        )
        for (tr, te), s in zip(splits, run_seeds)
    )
    timings["runs_s"] = round(time.perf_counter() - t, 1)

    ok = [r for r in results if np.isfinite(r["metric"])]
    for r in results:
        issues += r["issues"]
    if len(ok) < 2:
        log.error("fewer than 2 successful runs — skipped")
        return None
    if len(ok) < len(results):
        log.warning("    %d/%d runs failed (see issues)", len(results) - len(ok), len(results))

    metrics = np.array([r["metric"] for r in ok])
    y_true_all = np.concatenate([r["y_true"] for r in ok])
    y_pred_all = np.concatenate([r["y_pred"] for r in ok])
    pooled = score_metric(cfg.task, y_true_all, y_pred_all)
    mean_metric = float(metrics.mean())

    log.info("4/6 confidence intervals %s", list(cfg.ci_methods))
    cis: dict = {}
    if "runs" in cfg.ci_methods:
        t = time.perf_counter()
        _, cis["runs"] = ci_runs(metrics, cfg, spawn_seeds(cfg.seed + 1, cfg.n_boot))
        timings["ci_runs_s"] = round(time.perf_counter() - t, 1)
    if "predictions" in cfg.ci_methods:
        t = time.perf_counter()
        _, cis["predictions"] = ci_predictions(
            y_true_all, y_pred_all, cfg, spawn_seeds(cfg.seed + 2, cfg.n_boot)
        )
        timings["ci_predictions_s"] = round(time.perf_counter() - t, 1)
    if "refit" in cfg.ci_methods:
        t = time.perf_counter()
        if cfg.refit_tune:
            fixed = None
        elif cfg.tune:
            fixed = consensus_params([r["best_params"] for r in ok])
        else:
            fixed = dict(DEFAULT_PARAMS)
        _, cis["refit"], info = ci_refit(data, cfg, fixed, spawn_seeds(cfg.seed + 3, cfg.n_refit))
        if info["n_failed"]:
            issues.append(f"refit bootstrap: {info['n_failed']}/{info['n_draws']} draws failed")
        timings["ci_refit_s"] = round(time.perf_counter() - t, 1)

    log.info("5/6 permutation test")
    t = time.perf_counter()
    obs, nulls, pval = permutation_test(ok, cfg, spawn_seeds(cfg.seed + 4, cfg.n_perm))
    timings["permutation_s"] = round(time.perf_counter() - t, 1)

    log.info("6/6 feature importance")
    importance_top = importance_matrix = None
    if cfg.importance:
        t = time.perf_counter()
        importance_top, importance_matrix = aggregate_importance(
            [r["gain"] for r in ok], data.feature_names, cfg,
            spawn_seeds(cfg.seed + 5, cfg.n_boot_importance),
        )
        timings["importance_s"] = round(time.perf_counter() - t, 1)

    timings["total_s"] = round(time.perf_counter() - t0, 1)
    metric_name = "AUC" if cfg.task == "binary" else "C-index"
    log.info(
        "done in %.1fs — mean %s %.3f, pooled %.3f, permutation p=%.4f",
        timings["total_s"], metric_name, mean_metric, pooled, pval,
    )

    return {
        "target": target,
        "task": cfg.task,
        "cap_age": cfg.cap_age,
        "metric_name": metric_name,
        "n_rows": int(len(data.y)),
        "n_pairs": int(data.n_pairs),
        "mean_metric": mean_metric,
        "pooled_metric": float(pooled),
        "ci": cis,
        "pval": pval,
        "perm_null": nulls,
        "runs": pd.DataFrame(
            [{"run": i, "metric": r["metric"], "n_train": r["n_train"], "n_test": r["n_test"]}
             for i, r in enumerate(ok)]
        ),
        "best_params": [r["best_params"] for r in ok],
        "y_true": y_true_all,
        "y_pred": y_pred_all,
        "importance_top": importance_top,
        "importance_matrix": importance_matrix,
        "issues": issues,
        "timings": timings,
        "config": asdict(cfg),
    }

## 12 · Batch runner

`run_batch` loops over traits × age caps, **checkpoints to CSV after each combination**,
and skips completed rows on rerun (crash-safe resume). One batch per task type
(binary traits together, continuous traits together).

In [ ]:
def _summary_row(res: dict) -> dict:
    row = {
        "target": res["target"],
        "cap_age": res["cap_age"],
        "task": res["task"],
        "n_pairs": res["n_pairs"],
        "n_rows": res["n_rows"],
        "mean_metric": res["mean_metric"],
        "pooled_metric": res["pooled_metric"],
        "pval": res["pval"],
        "total_s": res["timings"]["total_s"],
    }
    for name in ("runs", "predictions", "refit"):
        lo, hi = res["ci"].get(name, (None, None))
        row[f"ci_{name}_lo"] = lo
        row[f"ci_{name}_hi"] = hi
    return row


def run_batch(
    df: pd.DataFrame,
    *,
    targets: list[str],
    age_steps: list,
    base_cfg: PipelineConfig,
    output_csv: str | None = None,):
    """Run the pipeline over targets x age caps. Checkpoints to CSV after each
    combination; rerunning skips rows already present in the CSV."""
    rows: list[dict] = []
    results: dict = {}
    done: set = set()
    if output_csv and os.path.exists(output_csv):
        old = pd.read_csv(output_csv)
        rows = old.to_dict("records")
        caps = [None if pd.isna(c) else c for c in old["cap_age"]]  # CSV stores None as NaN
        done = set(zip(old["target"], caps))
        log.info("resume: %d completed rows found in %s", len(done), output_csv)

    for target in targets:
        for cap in age_steps:
            if (target, cap) in done:
                log.info("skip %s cap_age=%s (already done)", target, cap)
                continue
            cfg = replace(base_cfg, cap_age=cap)
            log.info("=== %s | cap_age=%s ===", target, cap)
            try:
                res = run_pipeline(df, cfg, target)
            except Exception as e:  # noqa: BLE001 - one failure must not kill the batch
                log.error("%s cap_age=%s failed: %r", target, cap, e)
                continue
            if res is None:
                continue
            results[(target, cap)] = res
            rows.append(_summary_row(res))
            if output_csv:
                pd.DataFrame(rows).to_csv(output_csv, index=False)
                log.info("checkpointed -> %s", output_csv)

    return pd.DataFrame(rows), results

## RUN

In [ ]:
df = pd.read_csv("residualised_by_birth_day.csv", low_memory=False)
#df = df.dropna(subset=["sex"])
#df["random_binary"] = np.random.randint(0, 2, size=len(df))

# one trait, one age cap
#cfg = PipelineConfig(task="binary", seed=42, cap_age=None)   # task="continuous" for C-index traits
#res = run_pipeline(df, cfg, target="y")